# vit_small × {FunnyBirds, dSprites, ColoredMNIST} — XAI baselines

Companion to `walkthrough.ipynb`. Two parts:

1. **Train a small ViT** — the project's finetune CLI applied to
   the 22 M-parameter `vit_small_patch16_224.augreg_in21k_ft_in1k`
   on each of three datasets. Each dataset gets one short cell
   that shells out to `uv run python -m experiments.train_probe`.
   Skip a cell if the checkpoint already exists.
2. **Inspect with two published baselines** — once a checkpoint
   is trained, point the LeGrad and Chefer cells at it and look
   at heatmaps. Pick the layer / block of interest at the top of
   each cell. The point is to compare these well-known methods
   against our AttnLRP/CRP attributions on a model whose ground
   truth (what the model recognises) we control via the choice
   of training set.

Why these three datasets:

| Dataset | What you control | XAI use |
|---|---|---|
| **FunnyBirds**   | per-part ground truth | does the heatmap localise the right bird parts? |
| **dSprites**     | shape / scale / position factors | does the heatmap track the shape pixels? |
| **ColoredMNIST** | colour↔digit correlation, broken at test time | **intentionally biased** — model learns colour shortcut. Train-val ~ 0.98, test ~ 0.13. Heatmaps should reveal colour focus, not stroke geometry. |

## 1. Setup

In [ ]:
# %cd ../..
# %ls
from __future__ import annotations
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').is_file():
    REPO_ROOT = REPO_ROOT.parent

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from experiments.datasets import load as load_dataset
from experiments.models import BASES, build_probe
from experiments.viz_unfolded import to_display

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RUNS_DIR = REPO_ROOT / 'data' / 'runs'
print(f'device   : {DEVICE}')
print(f'runs dir : {RUNS_DIR}')

## 2. Train vit_small on each dataset

Three short cells, one per dataset. Each spells out the recipe in
a comment and shells out to the project's `finetune` CLI with
`--from-scratch`. All three write a timestamped run directory under
`data/runs/finetune_vit_small_<dataset>/<UTC ts>/` containing:

* `best.pt` — best-by-val_acc backbone + head + metadata
* `config.json` — full Typer params (reproducibility)
* `metrics.csv` — per-epoch logs (Lightning CSVLogger)

Hyperparams are tunable; if a run misses the > 96 % accuracy
target, raise `--layerwise-lr-decay`, lengthen
`--cosine-warmup-epochs`, or grow `--epochs`. The defaults
below match the recipe in the project plan.

Skip any cell whose checkpoint already exists (find with
`ls data/runs/finetune_vit_small_*/`).

### 2a. FunnyBirds (50-class part-based bird classification, ~1 h on a single GPU)

**Exact recipe that hit val_acc 0.9715 / test 0.9840.** OneCycleLR
(Smith SuperConvergence) with peak `backbone_lr=5e-4` and
`head_lr=5e-3` — ~50× higher than a cosine recipe. The OneCycle
ramp lets the head and late blocks adapt fast, then anneals to
convergence in 25 epochs. We tried cosine first and capped at
val_acc ≈ 0.89 over 50 epochs; OneCycle is the validated recipe.

`--train-ds funny-birds-train-clean` selects the ~29 k intact-bird
subset (drops the ~21 k part-ablation samples); use
`--train-ds funny-birds-train-full` to include them. `--num-workers 0`
if your `/dev/shm` is small (containers cap at 64 MB) — raise on
a wider box.

In [ ]:
!uv run python -m experiments.train_probe finetune \
    --from-scratch \
    --base vit_small \
    --head linear \
    --train-ds funny-birds-train-clean \
    --epochs 25 \
    --patience 25 \
    --backbone-lr 5e-4 \
    --head-lr 5e-3 \
    --weight-decay 0.05 \
    --batch-size 64 \
    --accumulate-grad-batches 2 \
    --layerwise-lr-decay 0.7 \
    --scheduler onecycle \
    --onecycle-pct-start 0.1 \
    --randaugment \
    --label-smoothing 0.1 \
    --val-frac 0.1 \
    --num-workers 0 \
    --seed 0

### 2b. dSprites — 3-class shape (square / ellipse / heart, ~30 min)

Synthetic, no augmentation needed. The 3-class shape task is
essentially trivial for a vit_small from in21k init.

In [ ]:
!uv run python -m experiments.train_probe finetune \
    --from-scratch \
    --base vit_small \
    --head linear \
    --train-ds dsprites \
    --dsprites-n-per-class 5000 \
    --epochs 10 \
    --patience 3 \
    --backbone-lr 1e-5 \
    --head-lr 1e-4 \
    --batch-size 128 \
    --accumulate-grad-batches 1 \
    --no-augment \
    --scheduler cosine \
    --cosine-warmup-epochs 2 \
    --num-workers 0

### 2c. ColoredMNIST — 10-class digit, colour↔digit 99 % correlation (~10 min)

**Intentionally biased model.** Geometric augmentation only — no
`--colorjitter-hue`. The model learns the colour shortcut: train-val
~ 0.98, test (uncorrelated colours) ~ 0.13. That mismatch is the
*goal* of this dataset — a model with a known, specific failure mode
lets us check whether XAI methods correctly localise the colour
pixels rather than the stroke. For a shape-aware ablation, add
`--colorjitter-hue 0.5` to randomise hue at train time.

In [ ]:
!uv run python -m experiments.train_probe finetune \
    --from-scratch \
    --base vit_small \
    --head linear \
    --train-ds colored-mnist-train \
    --epochs 20 \
    --patience 5 \
    --backbone-lr 1e-5 \
    --head-lr 3e-4 \
    --batch-size 128 \
    --accumulate-grad-batches 1 \
    --augment \
    --label-smoothing 0.1 \
    --scheduler cosine \
    --cosine-warmup-epochs 2 \
    --num-workers 0

## 3. Load a trained checkpoint

Set `CKPT` to the run you want to inspect. The default below
globs the most recent FunnyBirds run; change the dataset name in
the glob to switch.

In [ ]:
import json, glob

# Pick a train-ds and the most recent run for it. The train-ds
# string must match a key from `experiments.train_probe.TRAIN_DATASETS`.
TRAIN_DS = 'funny-birds-train-clean'   # 'funny-birds-train-clean' | 'dsprites' | 'colored-mnist-train'
_runs = sorted(glob.glob(str(RUNS_DIR / f'finetune_vit_small_{TRAIN_DS}' / '*' / 'best.pt')))
if not _runs:
    raise FileNotFoundError(
        f'No checkpoint found under {RUNS_DIR}/finetune_vit_small_{TRAIN_DS}/. '
        f'Re-run the matching training cell in section 2 first.'
    )
CKPT = Path(_runs[-1])
print(f'checkpoint: {CKPT}')
print(f'config    : {(CKPT.parent / "config.json").read_text()[:300]} ...')

In [ ]:
ckpt = torch.load(CKPT, map_location=DEVICE, weights_only=False)
model = build_probe(
    base=ckpt['base'], head=ckpt['head'],
    num_classes=ckpt['num_classes'],
    head_kwargs=ckpt.get('head_kwargs', {}),
).to(DEVICE)
model.backbone.load_state_dict(ckpt['backbone_state_dict'])
model.head.load_state_dict(ckpt['head_state_dict'])
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

print(f'val_acc  : {ckpt["val_acc"]:.4f}')
print(f'val_loss : {ckpt["val_loss"]:.4f}')
print(f'embed_dim: {model.backbone.embed_dim}, depth: {len(model.backbone.blocks)}, heads: {model.backbone.blocks[0].attn.num_heads}')

# Per-batch normalize closure (datasets emit unnormalized [0,1]).
from experiments.models import build_base
_base = build_base(ckpt['base'])
normalize = _base.get_normalize()
transform = _base.get_transform()

### Sanity-eval the loaded checkpoint on the held-out test split

For dSprites and FunnyBirds the test-split accuracy should match
the train-val number printed above. For **ColoredMNIST the test
accuracy is meant to collapse to ~10 %** — the model learned the
colour shortcut on train and the test split breaks the colour↔digit
correlation. That gap is the intended diagnostic, not a bug.

In [ ]:
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# Per-train-ds test set spec. dsprites has no held-out test split,
# so we just re-evaluate on its full training universe (the model
# trained on a subsample so this still measures generalisation).
_TEST_KWARGS = {
    'funny-birds-train-clean':   ('funny_birds',   dict(split='test')),
    'funny-birds-train-full':    ('funny_birds',   dict(split='test')),
    'colored-mnist-train':       ('colored_mnist', dict(split='test')),
    'dsprites':                  ('dsprites',      dict(target='shape')),
}
_test_name, _test_kwargs = _TEST_KWARGS[TRAIN_DS]
ds_test = load_dataset(_test_name, transform=transform, **_test_kwargs)
n_eval = min(2000, len(ds_test))
_loader = DataLoader(
    Subset(ds_test, list(range(n_eval))),
    batch_size=64, shuffle=False, num_workers=0,
)
_correct = _total = 0
with torch.no_grad():
    for _x, _y in tqdm(_loader, desc=f'test-eval ({TRAIN_DS})', unit='batch'):
        _logits = model(normalize(_x.to(DEVICE)))
        _correct += (_logits.argmax(-1).cpu() == _y).sum().item()
        _total   += _y.numel()
print(f'test top-1 over {_total} samples: {_correct/_total:.4f}')
print(f'  (train-val reported in ckpt: {ckpt["val_acc"]:.4f})')

In [ ]:
# Pick a focal image: first correctly-classified test sample.
_focal_name, _focal_kwargs = _TEST_KWARGS[TRAIN_DS]
ds = load_dataset(_focal_name, transform=transform, **_focal_kwargs)

focal_image = focal_class = focal_index = None
for i in range(0, len(ds), max(1, len(ds) // 50)):
    x_, y_ = ds[i]
    x_dev = x_.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = model(normalize(x_dev)).argmax(-1).item()
    if pred == int(y_):
        focal_image = x_dev
        focal_class = pred
        focal_index = i
        break
if focal_image is None:
    raise RuntimeError('no correctly-classified sample found in first 50 strided samples')

print(f'focal: index {focal_index}, class {focal_class}')
fig, ax = plt.subplots(1, 1, figsize=(3, 3))
ax.imshow(to_display(focal_image)); ax.axis('off')
ax.set_title(f'class {focal_class}')
plt.show()

## 4a. LeGrad — feature formation sensitivity

Bousselham et al., arXiv 2404.03214. Computes a heatmap by
differentiating the prediction w.r.t. the *attention weights* of
a chosen set of blocks, then aggregating per-token sensitivities
back to image space. Designed for ViTs. Library:
[`legrad_torch`](https://github.com/WalBouss/LeGrad).

The library's public wrapper expects an OpenCLIP `model.visual`
interface, so we hook directly via the lower-level method: the
library only needs to know which attention modules to attach to
and how to pull out the model logits. We pass our timm
`model.backbone.blocks[*].attn` list and the head as the logit
source.

**TARGET_BLOCKS** at the top of the cell controls which blocks'
attention weights contribute to the heatmap — change to inspect
shallow vs deep evidence. The default (all blocks) matches the
paper's recommended setting for ViT classification.

In [ ]:
# === LeGrad — pick which blocks contribute ===
TARGET_BLOCKS = list(range(len(model.backbone.blocks)))

# Stock timm `Attention.forward` calls `attn.softmax(dim=-1)` inline
# (or hides it inside `F.scaled_dot_product_attention` if `fused_attn`
# is True), so there is no `.softmax` submodule to hook. Apply our
# substitution canonizer first: it swaps Attention → TimmAttentionUnfolded
# (and EvaAttention → EvaAttentionUnfolded for DINOv3), exposing every
# tensor op — including `softmax` — as a named submodule. Forward
# output is bit-identical; `.remove()` restores the originals.
from zennit_ext import (
    TimmAttentionSubstitutionCanonizer,
    EvaAttentionSubstitutionCanonizer,
)
_canons = [TimmAttentionSubstitutionCanonizer(), EvaAttentionSubstitutionCanonizer()]
for c in _canons:
    c.apply(model)

_attn_maps = []   # one per block in forward order
_handles = []
def _capture_attn(_mod, _inp, _out):
    _attn_maps.append(_out)   # (B, num_heads, N, N), softmax output
for i in TARGET_BLOCKS:
    _handles.append(
        model.backbone.blocks[i].attn.softmax.register_forward_hook(_capture_attn)
    )

try:
    # Forward + backward through one-hot logit.
    for p in model.parameters():
        p.requires_grad_(False)
    x = normalize(focal_image).detach().clone().requires_grad_(False)
    _attn_maps.clear()
    logits = model(x)
    for a in _attn_maps:
        a.retain_grad()
    logits[0, focal_class].backward()

    # LeGrad heatmap: per-block, take grad of the attn map, average
    # over heads, keep cls-row, ReLU, then aggregate across blocks (mean).
    # The patch-token portion of cls-row reshapes to a square spatial map.
    npt = int(getattr(model, 'num_prefix_tokens', 1))
    per_block = []
    for a in _attn_maps:
        if a.grad is None:
            continue
        g = a.grad                          # (1, H, N, N)
        per = (g * a).clamp_min(0).mean(dim=1)   # (1, N, N) — head-mean
        cls_row = per[0, 0, npt:]                # (P,)
        per_block.append(cls_row)
    if not per_block:
        raise RuntimeError('no gradients captured — ensure focal_image was the model input')
    legrad_score = torch.stack(per_block).mean(dim=0)     # (P,)
    side = int(legrad_score.numel() ** 0.5)
    heatmap = legrad_score.reshape(side, side).detach().cpu().numpy()
finally:
    for h in _handles:
        h.remove()
    for c in _canons:
        c.remove()

fig, (ax_img, ax_hm) = plt.subplots(1, 2, figsize=(6, 3))
ax_img.imshow(to_display(focal_image)); ax_img.axis('off')
ax_img.set_title(f'class {focal_class}', fontsize=9)
vmax = abs(heatmap).max() or 1.0
ax_hm.imshow(heatmap, cmap='seismic', vmin=-vmax, vmax=vmax)
ax_hm.axis('off'); ax_hm.set_title(f'LeGrad — blocks {TARGET_BLOCKS}', fontsize=9)
plt.tight_layout(); plt.show()

## 4b. Chefer's method — gradient-weighted attention rollout

Chefer, Gur, Wolf. CVPR 2021, arXiv 2012.09838. Per block:
compute relevance from `(grad ⊙ A).clamp(0)` averaged over heads,
then roll up across blocks via $R \leftarrow R + R_{block} \cdot R$
starting from $R = I$. Code reference:
[`hila-chefer/Transformer-Explainability`](https://github.com/hila-chefer/Transformer-Explainability)
(`baselines/ViT/ViT_explanation_generator.py`). Vendored inline
because the upstream package isn't on PyPI and the rollout is short.

**UP_TO_BLOCK** at the top of the cell controls how deep the
rollup goes — set to a smaller integer to inspect the relevance
as it accumulates through the stack. Default = all blocks.

In [ ]:
# === Chefer's rollout — pick how many blocks to roll up ===
UP_TO_BLOCK = len(model.backbone.blocks) - 1   # 0-indexed inclusive

# Same fix as the LeGrad cell: substitute attention with its unfolded
# form so `.softmax` is a hookable named submodule.
from zennit_ext import (
    TimmAttentionSubstitutionCanonizer,
    EvaAttentionSubstitutionCanonizer,
)
_canons = [TimmAttentionSubstitutionCanonizer(), EvaAttentionSubstitutionCanonizer()]
for c in _canons:
    c.apply(model)

_attn_maps = []
_handles = []
def _capture_attn(_mod, _inp, _out):
    _attn_maps.append(_out)
for i in range(UP_TO_BLOCK + 1):
    _handles.append(
        model.backbone.blocks[i].attn.softmax.register_forward_hook(_capture_attn)
    )

try:
    x = normalize(focal_image).detach().clone()
    _attn_maps.clear()
    logits = model(x)
    for a in _attn_maps:
        a.retain_grad()
    logits[0, focal_class].backward()

    # Build per-block relevance R_block = E_h[(grad ⊙ A)+]
    blocks_R = []
    for a in _attn_maps:
        if a.grad is None:
            continue
        # Per Chefer Eq. 5: average over heads, ReLU, ignore neg.
        Rb = (a.grad * a).clamp_min(0).mean(dim=1)[0]   # (N, N)
        blocks_R.append(Rb)
    if not blocks_R:
        raise RuntimeError('no gradients captured')

    # Roll up: R = I; for each block, R = R + Rb @ R (Chefer Eq. 6).
    N = blocks_R[0].size(0)
    R = torch.eye(N, device=blocks_R[0].device)
    for Rb in blocks_R:
        R = R + Rb @ R

    # Pull cls-token row, drop prefix tokens, reshape to spatial.
    npt = int(getattr(model, 'num_prefix_tokens', 1))
    cls_row = R[0, npt:]
    side = int(cls_row.numel() ** 0.5)
    heatmap = cls_row.reshape(side, side).detach().cpu().numpy()
finally:
    for h in _handles:
        h.remove()
    for c in _canons:
        c.remove()

fig, (ax_img, ax_hm) = plt.subplots(1, 2, figsize=(6, 3))
ax_img.imshow(to_display(focal_image)); ax_img.axis('off')
ax_img.set_title(f'class {focal_class}', fontsize=9)
vmax = abs(heatmap).max() or 1.0
ax_hm.imshow(heatmap, cmap='seismic', vmin=-vmax, vmax=vmax)
ax_hm.axis('off')
ax_hm.set_title(f'Chefer rollout — blocks 0..{UP_TO_BLOCK}', fontsize=9)
plt.tight_layout(); plt.show()

## 5. Notes

* **Attention site.** Both methods hook `attn.softmax`, which is
  the post-softmax attention weight tensor `(B, H, N, N)` in
  both timm `Attention` and our `TimmAttentionUnfolded`. No
  composite needed — these are stock-model methods.
* **Why no statistics.** This notebook is for *manual* sanity
  checks: "does the heatmap look reasonable for this image?".
  Quantitative comparison (faithfulness, robustness, etc.) is a
  separate exercise; see Quantus / SaCo for harnesses.
* **Comparing to our AttnLRP/CRP.** Open `walkthrough.ipynb`
  side-by-side, set its section 2 to the same `(BASE, HEAD, DATASET)`
  — it auto-finds the same OneCycle checkpoint trained above and
  builds the FV index under `data/fv_cache/<base>_<head>_<dataset>/`.
  Caches are keyed per-combination so vit_small/dsprites doesn't
  collide with vit_small/funny_birds; multiple kernels with different
  selections can index in parallel without interfering.
* **ColoredMNIST is the bias probe.** The trained checkpoint is
  *meant* to score ~10 % on the test split (uncorrelated colours)
  — it learned colour, not digit shape. That's the **target**,
  not a bug. Run the LeGrad / Chefer cells on the cmnist
  checkpoint with `DATASET = 'colored_mnist'` and check whether
  the heatmaps fire on the *whole* coloured silhouette (colour
  shortcut detected — XAI is working) or on stroke geometry
  (XAI is mis-identifying the cause of the prediction).